# 01 — Analytical setpoint (LT3010-5 datasheet example)

Fixed LT3010-5, datasheet Rev. J page 1 typical application "5V Supply with Shutdown": 5 V at 50 mA, 1 µF input and output capacitors, VIN printed as 5.4 V to 80 V. Single-point calculations use 12 V, which is inside that range. Page 3 is the source of every IC min/typ/max used below.

## Datasheet example

Page 1, figure TA01, "5V Supply with Shutdown":

| Item | Value on the figure |
|------|---------------------|
| Part | LT3010-5 |
| VIN | 5.4 V to 80 V |
| VOUT | 5 V |
| Load | 50 mA |
| Input capacitor | 1 µF |
| Output capacitor | 1 µF |
| SHDN | above 2.0 V on, below 0.3 V off |

SENSE is tied to OUT, so there is no external divider. The page-3
over-temperature output box (4.850–5.150 V) already includes line, load,
and temperature. Default WC/MC uses that envelope and does not stack
line and load on top of it.

## Setup — editable BOM and run knobs

This cell is shared by every notebook. It puts `P5V_Regulator/` on `sys.path`
and builds **one** `CircuitParams` object. Change a resistor, capacitor, or
tolerance here and re-run: analytical, SPICE, WC, and MC all follow this object.

**Plot callouts:** each figure cell has an `ann = [dict(...), ...]` list.
Edit `text`, `xy` (data coords of the arrow tip), and `offset=(dx, dy)` in
**points** to nudge a label. Use `xytext=(x, y)` instead of `offset` for data
coordinates. `arrow=False` for a plain label. `ann = []` hides all callouts.

Tolerances are **fractions** (0.01 = 1 %). Figures go to `results/figures/`.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def _find_project_root() -> Path:
    start = Path.cwd().resolve()
    named = Path("Sections/Electronics/Analog_and_Digital/LT3010_5V_Supply")
    candidates = [start, *start.parents, start / named]
    if start.name == "notebooks":
        candidates.insert(0, start.parent)
    for cand in candidates:
        if (cand / "p5v_design" / "__init__.py").is_file():
            return cand
    return start

ROOT = _find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from p5v_design.params import CircuitParams, ToleranceSpec, format_eng, LdoParams
from p5v_design import analysis, schematic, plots, stability
from p5v_design import plots_wc_mc as pwc
from p5v_design.ldo_model import dropout_v, ignd_a, tj_c, psrr_db, vadj_typical_vs_temp
from dataclasses import replace

# --- EDITABLE BOM ---
# Datasheet TA01, "5V Supply with Shutdown" (LT3010/LT3010-5 Rev. J page 1).
# Fixed 5 V part: SENSE tied to OUT (R1 = 0). No external divider (R2 open).
# Capacitors are the 1 µF parts drawn on that figure. Load is the labeled 50 mA.
# VIN is printed as 5.4 V to 80 V; 12 V is the single-point bias inside that range.
R1 = 0.0             # SENSE shorted to OUT
R2 = 1.0e12          # open — no bottom resistor
CFF = 0.0            # no feedforward capacitor on the fixed part
COUT = 1.0e-6
CIN = 1.0e-6
ESR_OUT = 10e-3      # ceramic assumption, not printed on TA01
ESR_IN = 10e-3
RLOAD = 100.0        # 5 V / 50 mA
R_TOL = 0.01         # 1 %
C_TOL = 0.10         # 10 %
VIN_NOM = 12.0
VIN_MIN = 5.4
VIN_MAX = 80.0
ILOAD_OP = 50e-3
N_MC = 2000
N_SPICE_MC = 40
N_LTSPICE_MC = 200
SKIP_SPICE_WC_MC = False
RUN_LTSPICE_BATCH = False
SPICE_WORKERS = None
LTSPICE_WORKERS = None

p = CircuitParams(
    r1=R1, r2=R2, cff=CFF, cout=COUT, cin=CIN,
    esr_out=ESR_OUT, esr_in=ESR_IN, r_load=RLOAD,
    r_tol=ToleranceSpec(R_TOL), c_tol=ToleranceSpec(C_TOL),
)
p = replace(p, op=replace(p.op, vin_nom=VIN_NOM, vin_min=VIN_MIN, vin_max=VIN_MAX,
                           iload_op=ILOAD_OP, r_load=RLOAD))

FIG = ROOT / "results" / "figures"
FIG.mkdir(parents=True, exist_ok=True)
print("gain =", p.gain, "  Vout nom =", analysis.vout_dc(p))
print("SPICE_WORKERS =", SPICE_WORKERS, "  N_MC =", N_MC)


gain = 1.0   Vout nom = 5.0
SPICE_WORKERS = None   N_MC = 2000


## Schematic and nominal setpoint

SENSE is tied to OUT, so the typical setpoint is the page-3 5.000 V, before IC and resistor tolerances. That bias is reported, not “fixed”.

In [2]:
from IPython.display import display, Markdown
sp = analysis.setpoint(p)
tbl = analysis.setpoint_table(p)
display(analysis.style_metrics(tbl))
display(Markdown("$$" + analysis.latex_vout() + "$$"))
print(f"setpoint bias vs 5.000 V: {1e3*sp['setpoint_bias']:+.2f} mV  ({sp['setpoint_bias_pct']:+.2f} %)")
sch = schematic.draw_schematic(FIG / "01_schematic.png", p)
print("saved", sch)

,value,unit
parameter,,
vadj,5,V
gain,1,—
r1,0,kΩ
r2,1.0000e+09,kΩ
iadj,0,nA
i_div,5.0000e-09,mA
vout,5,V
vout_no_iadj,5,V
dv_iadj,0,mV


$$V_{\mathrm{OUT}}=V_{\mathrm{ADJ}}\left(1+\frac{R_1}{R_2}\right)+I_{\mathrm{ADJ}}R_1$$

setpoint bias vs 5.000 V: +0.00 mV  (+0.00 %)


saved results/figures/01_schematic.png


## Divider current vs $I_{\mathrm{ADJ}}$

Datasheet: keep the bottom resistor under 250 kΩ so $I_{\mathrm{ADJ}}$ is a small error. Our R2 = 2.2 kΩ is comfortably below that.

In [3]:
stab = stability.stability_table(p)
display(stab)
ann_div = [
    dict(text="operate Idiv", xy=(p.r2, 1e6 * p.i_div), offset=(18, 12), arrow=True),
]
# Idiv vs R2 family
r2g = np.logspace(3, 5.4, 80)
idiv = p.vadj / r2g
fig, ax = plt.subplots(figsize=(8.0, 4.2))
ax.loglog(r2g, 1e6 * idiv, "k-", lw=1.8, label=r"$I_{div}=V_{ADJ}/R_2$")
ax.axhline(p.ldo.iadj_typ * 1e6, color="C3", ls="--", lw=1.1, label="Iadj typ 50 nA")
ax.axhline(p.ldo.iadj_max * 1e6, color="C1", ls=":", lw=1.1, label="Iadj max 100 nA")
ax.axvline(p.r2, color="0.45", ls="--")
ax.axvline(250e3, color="C4", ls="-.", lw=1.0, label="250 kΩ guideline")
ax.set_xlabel(r"$R_2$ (Ω)")
ax.set_ylabel(r"$I_{div}$ (µA)")
ax.set_title("Feedback divider current vs bottom resistor")
ax.legend(fontsize=8)
ax.grid(True, which="both", alpha=0.3)
plots.apply_annotations(ax, ann_div)
plots.savefig(fig, FIG / "01_idiv_vs_r2.png")
plt.show()

,check,value,unit,note
0,Divider current,5.000000e-12,A,>> Iadj
1,Idiv / Iadj typ,5.000000e+18,,> 100 recommended
2,R2 < 250 kΩ,0.000000e+00,,datasheet Iadj guideline
3,Cout ≥ 1 µF,1.000000e+00,,min for stability
4,ESR ≤ 3 Ω,1.000000e+00,,datasheet
5,|Xcff| @ 10 kHz,inf,Ω,must be < R2 = 1e+12 Ω
6,Cff rule,0.000000e+00,,Datasheet Fig. 2: Z(Cff) at 10 kHz should be <...


cell:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Cff 10 kHz rule and Cout/ESR window

In [4]:
f, z, rbot = stability.cff_impedance_sweep(p)
ann_cff = [
    dict(text="10 kHz rule", xy=(10e3, rbot), offset=(12, 14), arrow=True),
    dict(text="|Xcff|", xy=(10e3, z[np.argmin(np.abs(f-10e3))]), offset=(12, -18), arrow=True),
]
fig, ax = plt.subplots(figsize=(8.0, 4.2))
ax.loglog(f, z, "k-", lw=1.8, label=r"$|X_{Cff}|$")
ax.axhline(rbot, color="C3", ls="--", label=f"R2 = {format_eng(rbot, 'Ω')}")
ax.axvline(10e3, color="0.45", ls=":")
ax.set_xlabel("f (Hz)")
ax.set_ylabel("Ω")
ax.set_title("Cff impedance vs datasheet 10 kHz rule")
ax.legend(fontsize=8)
ax.grid(True, which="both", alpha=0.3)
plots.apply_annotations(ax, ann_cff)
plots.savefig(fig, FIG / "01_cff_z.png")
plt.show()
print(f"Cout {format_eng(p.cout,'F')}  (min {format_eng(p.ldo.cout_min_f,'F')})   "
      f"ESR {format_eng(p.esr_out,'Ω')}  (max {p.ldo.esr_max_ohm} Ω)")

p5v_design/stability.py:33: RuntimeWarning: divide by zero encountered in divide
  z = 1.0 / (2.0 * np.pi * f * p.cff)
cell:9: UserWarning: Attempting to set identical low and high ylims makes transformation singular; automatically expanding.
  ax.axvline(10e3, color="0.45", ls=":")
p5v_design/plots.py:153: UserWarning: Attempting to set identical low and high xlims makes transformation singular; automatically expanding.
  fig.tight_layout()


Cout 1 µF  (min 1 µF)   ESR 10 mΩ  (max 3.0 Ω)


cell:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Page-3 error budget — envelope vs stacked

Envelope terms go into WC/MC. Stacked extras (line, load) are **already inside** the over-temp $V_{\mathrm{ADJ}}$ box.

In [5]:
bud_25 = analysis.error_budget_table(p, over_temp=False)
bud_ot = analysis.error_budget_table(p, over_temp=True)
print("25 °C envelope RSS half-span", f"{bud_25.attrs['rss_half_span_mV']:.2f} mV")
print("over-temp envelope RSS half-span", f"{bud_ot.attrs['rss_half_span_mV']:.2f} mV")
display(bud_25)
ann_bud = []  # tornado is self-labelled; add dicts here if you want callouts
plots.plot_error_budget(bud_25, annotations=ann_bud, path=FIG/"01_budget_envelope.png",
                        title="Envelope OAT half-span (25 °C)")
plt.show()
plots.plot_error_budget_signed(bud_25, annotations=[], path=FIG/"01_budget_signed.png")
plt.show()
env = analysis.envelope_bounds(p, over_temp=False)
env_ot = analysis.envelope_bounds(p, over_temp=True)
stk = analysis.stacked_bounds(p, over_temp=True)
print("envelope 25 °C  Vout", env)
print("envelope over temp", env_ot)
print("stacked (pessimistic)", stk)

25 °C envelope RSS half-span 75.00 mV
over-temp envelope RSS half-span 150.00 mV


,term,group,vout_lo,vout_hi,delta_hi_mV,delta_lo_mV,half_span_mV,note
0,Vadj box,envelope,4.925,5.075,75.0,-75.0,75.0,page 3 25 °C ADJ
1,Iadj,envelope,5.000,5.000,0.0,0.0,0.0,current into ADJ through R1
2,R1 (top),envelope,5.000,5.000,0.0,0.0,0.0,±1 %
3,R2 (bot),envelope,5.000,5.000,0.0,0.0,0.0,±1 %
4,Line (stacked extra),stacked,5.000,5.000,0.0,0.0,0.0,Do not add on top of over-temp Vadj box
5,Load (stacked extra),stacked,4.950,5.000,0.0,-50.0,25.0,G23: Vout falls as Iload rises


cell:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


envelope 25 °C  Vout {'nominal': 5.0, 'min': 4.925, 'max': 5.075, 'span': 0.15000000000000036, 'min_pct': -1.5000000000000013, 'max_pct': 1.5000000000000124}
envelope over temp {'nominal': 5.0, 'min': 4.85, 'max': 5.15, 'span': 0.3000000000000007, 'min_pct': -3.0000000000000027, 'max_pct': 3.0000000000000027}
stacked (pessimistic) {'nominal': 5.0, 'min': 4.827489932885906, 'max': 5.082510067114094, 'span': 0.2550201342281877, 'line_half_v': 0.00751006711409396, 'load_drop_v': 0.09}


cell:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## $V_{\mathrm{OUT}}$ vs $V_{\mathrm{IN}}$ (dropout knee + line)

In [6]:
vin, v_typ = analysis.vout_vs_vin(p, which_do="typ")
_, v_wc = analysis.vout_vs_vin(p, which_do="max_ot")
vdo_op = dropout_v(p, which="max_ot")
ann_vin = [
    dict(text="12 V bias", xy=(12.0, analysis.vout_dc(p)), offset=(16, -22), arrow=True, color="C3"),
    dict(text="dropout knee", xy=(5.0 + vdo_op, analysis.vout_dc(p) * 0.92), offset=(16, -18), arrow=True),
    dict(text="target 5 V", xy=(11.0, 5.0), offset=(0, 12), arrow=False),
]
plots.plot_vout_vs_vin(
    vin, v_typ, vout_wc=v_wc, vin_nom=p.op.vin_nom, vout_target=p.op.vout_target,
    vdo=vdo_op, annotations=ann_vin, path=FIG/"01_vout_vs_vin.png",
)
plt.show()
hr = analysis.headroom(p, which="max_ot")
display(analysis.style_metrics(analysis.metrics_table(hr)))
plots.plot_headroom_bar(p, annotations=[
    dict(text=f"headroom {hr['headroom']:.2f} V", xy=(p.op.vin_nom*0.72, 0.0), offset=(0, 18), arrow=False),
], path=FIG/"01_headroom.png")
plt.show()

cell:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,value,unit
parameter,,
vin,12,V
vout,5,V
iload,50,mA
vdo,550,mV
vin_min_reg,5.55,V
headroom,6.45,V
in_dropout,False,—


cell:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Load regulation, dropout, and $I_{\mathrm{GND}}$

In [7]:
i, vo = analysis.vout_vs_iload(p)
ann_load = [
    dict(text="50 mA load", xy=(50.0, analysis.vout_dc(p)), offset=(12, 12), arrow=True),
]
plots.plot_vout_vs_iload(i, vo, iload_op=p.op.iload_op, annotations=ann_load,
                         path=FIG/"01_vout_vs_i.png")
plt.show()
ann_do = [
    dict(text="50 mA", xy=(50.0, 1e3*dropout_v(p, which="typ")), offset=(10, 14), arrow=True),
    dict(text="max over temp", xy=(50.0, 550), offset=(-40, 10), arrow=True, color="C3"),
]
plots.plot_dropout_vs_i(p, annotations=ann_do, path=FIG/"01_dropout.png")
plt.show()
ann_ig = [
    dict(text="operate", xy=(35.0, 1e3*ignd_a(p, which="typ")), offset=(12, 12), arrow=True),
]
plots.plot_ignd_vs_i(p, annotations=ann_ig, path=FIG/"01_ignd.png")
plt.show()

cell:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


cell:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


cell:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Regulation families (datasheet G-curve style)

In [8]:
ann_f = [dict(text="12 V / 50 mA", xy=(12.0, analysis.vout_dc(p)), offset=(12, -18), arrow=True)]
plots.plot_regulation_family(p, which="vin", annotations=ann_f, path=FIG/"01_family_vin.png")
plt.show()
plots.plot_regulation_family(p, which="i", path=FIG/"01_family_i.png")
plt.show()

cell:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


cell:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## PSRR, noise, thermal, current-limit margin

In [9]:
ann_psrr = [
    dict(text="120 Hz spec 75 dB typ", xy=(120, 75), offset=(14, 8), arrow=True),
]
plots.plot_psrr(p, annotations=ann_psrr, path=FIG/"01_psrr.png")
plt.show()
ann_n = [
    dict(text=r"~1.5 µV/√Hz", xy=(1e3, 1.5), offset=(12, 10), arrow=True),
]
plots.plot_noise_psd(p, annotations=ann_n, path=FIG/"01_noise_psd.png")
plt.show()
th = tj_c(p, which="typ")
display(analysis.style_metrics(analysis.metrics_table(th)))
print(f"Ilim min over temp {1e3*p.ldo.ilim_min_ot_a:.0f} mA   operate {1e3*p.op.iload_op:.0f} mA   "
      f"margin {1e3*(p.ldo.ilim_min_ot_a-p.op.iload_op):.0f} mA")
m = analysis.summary_metrics(p)
display(analysis.style_metrics(analysis.metrics_table(m)))
ripple = analysis.psrr_ripple_out(p, vin_ripple_pp=0.05, f_hz=120.0)
print(f"50 mVpp @ 120 Hz on VIN → {1e3*ripple:.3f} mVpp on VOUT (datasheet PSRR curve)")

cell:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


cell:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,value,unit
parameter,,
vin,12,V
iload,50,mA
vout,5,V
ignd,1.8,mA
p_pass,350,mW
p_gnd,21.6,mW
p_tot,371.6,mW
rth_ja,40,°C/W
ta_c,25,°C


Ilim min over temp 60 mA   operate 50 mA   margin 10 mA


,value,unit
parameter,,
vout,5,V
vadj,5,V
gain,1,—
setpoint_bias,0,mV
i_div,5.0000e-09,mA
dv_iadj,0,mV
vin,12,V
iload,50,mA
vdo_typ,300,mV


50 mVpp @ 120 Hz on VIN → 0.018 mVpp on VOUT (datasheet PSRR curve)


## Extra datasheet families (from HV_monitor / Opamp / comparator notebooks)

PSRR 1 µF vs 10 µF (G21), integrated noise vs bandwidth, current-limit margin, headroom vs Vin, and the over-temp error-budget tornado.

In [10]:
ann_pf = [dict(text="BOM 10 µF", xy=(120, 75), offset=(14, 8), arrow=True)]
plots.plot_psrr_family(p, annotations=ann_pf, path=FIG/"01_psrr_family.png")
plt.show()
ann_nrms = [dict(text="10 kHz", xy=(1e4, 150), offset=(12, 10), arrow=True)]
plots.plot_noise_rms_vs_bw(p, annotations=ann_nrms, path=FIG/"01_noise_rms_bw.png")
plt.show()
plots.plot_ilim_margin(p, annotations=[
    dict(text="50 mA load", xy=(50, 0), offset=(12, 12), arrow=True),
], path=FIG/"01_ilim.png")
plt.show()
plots.plot_headroom_vs_vin(p, annotations=[
    dict(text="12 V bias", xy=(12.0, analysis.headroom(p)["headroom"]), offset=(12, 10), arrow=True),
], path=FIG/"01_headroom_vs_vin.png")
plt.show()
plots.plot_error_budget(bud_ot, annotations=[], path=FIG/"01_budget_overtemp.png",
                        title="Envelope OAT half-span (over temp)")
plt.show()
plots.plot_wc_vout_vs_vin(p, annotations=[
    dict(text="dropout knee", xy=(5.4, 4.7), offset=(14, -12), arrow=True),
], path=FIG/"01_wc_vs_vin.png")
plt.show()

cell:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


cell:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


cell:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


cell:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
cell:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


cell:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
